In [1]:
import os
import json
import polars as pl
from tqdm import tqdm
import cudf
import cupy as cp
from numba import cuda



import plotly.graph_objects as go
from plotly.subplots import make_subplots

mempool = cp.get_default_memory_pool()
mempool.set_limit(size=0.8 * 1024**3)  # Limite à 80% de la VRAM


In [5]:
# Fonction pour charger les données parquet
def load_parquet(stock, date):
    file_path = f'/home/janis/3A/EA/HFT_QR_RL/data/smash4/DB_MBP_10/{stock}/{stock}_{date}.parquet'
    return cudf.read_parquet(file_path)
# Spécifier les dates
dates = ["2024-08-12"]

stocks = ["CSX"]

# Charger les données pour chaque stock et chaque date
data_dict = {}
for stock in stocks:
    data_dict[stock] = {}
    for date in dates:
        # Use cudf's random sampling with fraction
        data_dict[stock][date] = load_parquet(stock, date).sample(frac=0.1, random_state=42)

# Concaténer toutes les données using cudf concat
data = cudf.concat([data_dict[stock][date] for stock in stocks for date in dates])

# Sort using cudf's sort_values
data = data.sort_values('ts_event')


In [6]:
# Calculate basic statistics for each stock and date
stats_dict = {}
for stock in stocks:
    stats_dict[stock] = {}
    for date in dates:
        df = data_dict[stock][date]
        
        # Calculate mid price
        mid_price = (df['bid_px_00'] + df['ask_px_00']) / 2
        
        # Calculate spread
        spread = df['ask_px_00'] - df['bid_px_00']
        
        stats = {
            'mean_mid_price': float(mid_price.mean()),
            'std_mid_price': float(mid_price.std()),
            'min_mid_price': float(mid_price.min()),
            'max_mid_price': float(mid_price.max()),
            'mean_spread': float(spread.mean()),
            'std_spread': float(spread.std()),
            'min_spread': float(spread.min()),
            'max_spread': float(spread.max()),
            'total_volume_bid': int(df['bid_ct_00'].sum()),
            'total_volume_ask': int(df['ask_ct_00'].sum()),
            'num_quotes': len(df),
        }
        
        stats_dict[stock][date] = stats

# Print statistics
for stock in stats_dict:
    print(f"\nStatistics for {stock}:")
    for date in stats_dict[stock]:
        print(f"\nDate: {date}")
        for metric, value in stats_dict[stock][date].items():
            if 'price' in metric:
                print(f"{metric}: ${value:.4f}")
            elif 'spread' in metric:
                print(f"{metric}: ${value:.6f}")
            else:
                print(f"{metric}: {value:,}")



Statistics for CSX:

Date: 2024-08-12
mean_mid_price: $33.5799
std_mid_price: $0.1479
min_mid_price: $33.3950
max_mid_price: $42.1750
mean_spread: $0.015481
std_spread: $0.326767
min_spread: $0.010000
max_spread: $39.600000
total_volume_bid: 240,229
total_volume_ask: 211,915
num_quotes: 20,882


In [15]:
# Create an interactive plotly figure
for date in dates:
    for stock in tqdm(stocks, desc="Processing stocks"):
        fig = go.Figure()
        
        # Get data for this stock and date and sort by timestamp
        df = load_parquet(stock, date).sort_values('ts_event')
        
        # Calculate mid price using pandas API
        df['mid_price'] = (df['bid_px_00'] + df['ask_px_00']) / 2

        # Add mid price line
        fig.add_trace(go.Scatter(
            x=df['ts_event'],
            y=df['mid_price'],
            mode='lines',
            name='Mid Price',
            line=dict(color='black', width=1)
        ))

        # Add best bid price with size-proportional markers
        fig.add_trace(go.Scatter(
            x=df['ts_event'],
            y=df['bid_px_00'],
            mode='lines+markers',
            name='Best Bid',
            line=dict(color='green', width=1),
            marker=dict(
                size=df['bid_sz_00'],
                sizeref=2.*df['bid_sz_00'].max()/17**2,
                sizemode='area',
                color='green',
                opacity=0.3
            )
        ))

        # Add best ask price with size-proportional markers
        fig.add_trace(go.Scatter(
            x=df['ts_event'],
            y=df['ask_px_00'],
            mode='lines+markers',
            name='Best Ask', 
            line=dict(color='red', width=1),
            marker=dict(
                size=df['ask_sz_00'],
                sizeref=2.*df['ask_sz_00'].max()/17**2,
                sizemode='area',
                color='red',
                opacity=0.3
            )
        ))

        # Add second best bid price
        fig.add_trace(go.Scatter(
            x=df['ts_event'],
            y=df['bid_px_01'],
            mode='lines',
            name='Second Best Bid',
            line=dict(color='rgba(0,255,0,0.3)', width=1)
        ))

        # Add second best ask price
        fig.add_trace(go.Scatter(
            x=df['ts_event'],
            y=df['ask_px_01'],
            mode='lines',
            name='Second Best Ask',
            line=dict(color='rgba(255,0,0,0.3)', width=1)
        ))

        # Add trades as black dots
        trades = df[df['rtype'] == 2]  # Assuming rtype 2 indicates trades
        fig.add_trace(go.Scatter(
            x=trades['ts_event'],
            y=trades['mid_price'],  # Using mid price for trade points
            mode='markers',
            name='Trades',
            marker=dict(color='black', size=8)
        ))

        # Update layout with fixed x and y ranges
        fig.update_layout(
            title=f"Order Book Visualization for {stock} on {date}",
            xaxis_title="Time",
            yaxis_title="Price",
            showlegend=True,
            width=1200,
            height=800,
            xaxis=dict(
                range=['13:00', '20:00']  # Set x-axis range from 13h to 20h
            ),
            yaxis=dict(
                range=[32, 36]  # Set y-axis range from 32 to 36
            )
        )

        # Create directory if it doesn't exist
        save_dir = f"/home/janis/3A/EA/HFT_QR_RL/data/smash4/plotly/{stock}"
        os.makedirs(save_dir, exist_ok=True)

        # Save the plot as HTML
        fig.write_html(f"{save_dir}/orderbook_visualization_{stock}_{date}.html")

        # Show the plot
        fig.show()


Processing stocks:   0%|          | 0/1 [00:00<?, ?it/s]


TypeError: Implicit conversion to a host NumPy array via __array__ is not allowed, To explicitly construct a GPU matrix, consider using .to_cupy()
To explicitly construct a host matrix, consider using .to_numpy().

: 